# Deriving Stefan Fuchs

For single component evaporation, the mass of molecules leaving a droplet exceeds the mass entering the droplet. Naively, you'd expect these quantities to scale with the droplet surface area. Instead, they end up scaling with the droplet radius instead. This is seen through the r2 evaporation law where the square radius decreases linearly in time for water droplets. This means the rate limiting step is not the rate of collision but instead something else.

We're going to start by making some assumptions:

1. There exists some steady film of gas surrounding the droplet
2. Transport within the film represents the rate limiting step
3. The concentration of gas at the droplet's surface is in equilibrium with the droplet. This is because the equilibration process is much faster than the film transport process.
4. The entire system is spherically symmetric, concentration and transport only varies as you travel away from the droplet. At a given distance, all concentrations are the same.

It stands to reason that N + 1 quantities must be preserved, where N is the number of chemical species:

1. The mass of each transported species (in our case this is the air + the water vapour)
2. The total energy within the system

# Continuity Equation

To preserve mass, the continuity equation is used. $\boldsymbol{j}$ is a symbol called the flux that represents the number of molecules passing through some small area every second. $\nabla \cdot$ is an operator, called the divergence, that finds out the difference between the number of molecules entering a small volume and those leaving.

To preserve mass, the divergence of the flux should be zero away from the droplet surface. There should be the same number of molecules entering a volume as leave it. At the droplet surface, the divergence should be equal to the net number of molecules entering the gas phase every second.

At the surface:

$
\nabla \cdot \boldsymbol{j} = \dot{m}
$

Away from the surface:

$
\nabla \cdot \boldsymbol{j} = 0
$


Molecules move due to bulk velocity, think wind, and by diffusion from a concentration gradient. Overall, $\boldsymbol{j}$ is made of just these two terms:

$
\boldsymbol{j} = \rho_g \boldsymbol{u} - D \rho_g \nabla Y
$

Note that the $\nabla$ without the dot above is the gradient of $Y$, a 3D equivalent of a derivative.


The divergence can be written in the following form in a radially symmetric system:

$
\nabla \cdot \boldsymbol{j} = \frac{1}{r^2} \frac{\delta (r^2 j_r)}{\delta r}
$

We can also write th gradient of the mass $Y$ as $\frac{\delta Y}{\delta r}$.

Let's use some python tools to do the rest of the derivation from here!

In [183]:
import sympy as sp
from IPython.display import display

gas_density, diffusion_coefficient, position = sp.symbols("rho_g D r")
gas_velocity = sp.Function("u_r")(position)
mass_fraction = sp.Function("Y")(position)

radial_flux = gas_density * gas_velocity * mass_fraction - diffusion_coefficient * gas_density * sp.Derivative(mass_fraction, position, evaluate=False)
display(radial_flux)

-D*rho_g*Derivative(Y(r), r) + rho_g*Y(r)*u_r(r)

In [184]:
divergence = 1/position**2 * sp.Derivative(position**2*radial_flux, position, evaluate=False)
display(divergence)

Derivative(r**2*(-D*rho_g*Derivative(Y(r), r) + rho_g*Y(r)*u_r(r)), r)/r**2

Now, to get this equal to zero, the structure inside the derivative must be a constant with radial position (meaning its gradient is zero). This interior is just the flux times the radial position squared. The flux times the area is the rate of change of mass.

If we multiply the interior by $4\pi$ to convert to spherical area, and evaluate terms at the droplet surface, this will be equal to the rate of change of mass of the droplet! As this is a constant, this expression will hold at any radial position as well.

In [185]:
time = sp.symbols("t")
mass = sp.Function("m")(time)
mass_derivative = mass.diff(time)

integrated_form = (4*sp.pi)*position**2*radial_flux
display(sp.Eq(integrated_form,mass_derivative))

Eq(4*pi*r**2*(-D*rho_g*Derivative(Y(r), r) + rho_g*Y(r)*u_r(r)), Derivative(m(t), t))

To help solve this equation, we can imagine what needs to happen to ALL species in the gas phase as a single medium. In that case, $Y(r)$ is equal to 1 everywhere and so its derivative is zero everywhere. This means we can express the mass derivative in terms of just the bulk gas velocity:

In [186]:
single_medium = integrated_form.subs(mass_fraction,1).doit() #Simplify the expression by taking Y as 1 for the entire medium
single_medium_equality = sp.Eq(single_medium,mass_derivative)
display(single_medium_equality)

gas_velocity_solve = sp.solve(single_medium_equality, gas_velocity)[0]
display(sp.Eq(gas_velocity,gas_velocity_solve))

Eq(4*pi*r**2*rho_g*u_r(r), Derivative(m(t), t))

Eq(u_r(r), Derivative(m(t), t)/(4*pi*r**2*rho_g))

Lets now sub in our new expression for the gas velocity into the integrated form above.

In [187]:
final_form = integrated_form.subs(gas_velocity,gas_velocity_solve)
final_form_eq = sp.Eq(final_form,mass_derivative)
final_form_solve = sp.solve(final_form_eq,mass_derivative)[0]
display(sp.Eq(mass_derivative,final_form_solve))

Eq(Derivative(m(t), t), 4*pi*D*r**2*rho_g*Derivative(Y(r), r)/(Y(r) - 1))

We have an interesting form that depends on this $\frac{\frac{dY}{dr}}{Y-1}$ term. This is the derivative of $\ln|Y-1|$ or better written as $\ln(1-Y)$. I'll let sympy do that solve though...

In [188]:
ode_solve = sp.dsolve(sp.Eq(mass_derivative,final_form_solve),mass_fraction)
display(ode_solve)

Eq(Y(r), C1*exp(-Derivative(m(t), t)/(4*pi*D*r*rho_g)) + 1)

We must now apply a pair of boundary conditions. At the droplet surface, Y = Y_s and far away from the droplet, Y = Y_inf

In [189]:
surface_mass_fraction, ambient_mass_fraction, droplet_radius, C1 = sp.symbols(r"Y_s Y_{\infty} r_s C1")
mass_fraction_expression = ode_solve.subs(position, droplet_radius).subs(mass_fraction.subs(position,droplet_radius), surface_mass_fraction)

constant_solve = sp.solve(mass_fraction_expression,C1)[0]
display(sp.Eq(C1,constant_solve))

new_ode_solve = ode_solve.subs(C1,constant_solve)
display(new_ode_solve)

Eq(C1, (Y_s - 1)*exp(Derivative(m(t), t)/(4*pi*D*r_s*rho_g)))

Eq(Y(r), (Y_s - 1)*exp(-Derivative(m(t), t)/(4*pi*D*r*rho_g))*exp(Derivative(m(t), t)/(4*pi*D*r_s*rho_g)) + 1)

In [190]:
far_field = sp.limit(new_ode_solve.rhs, position, sp.oo)
display(sp.Eq(ambient_mass_fraction,far_field))

answer = sp.solve(sp.Eq(ambient_mass_fraction,far_field),mass_derivative)[0]
display(sp.Eq(mass_derivative,answer))

Eq(Y_{\infty}, Y_s*exp(Derivative(m(t), t)/(4*pi*D*r_s*rho_g)) - exp(Derivative(m(t), t)/(4*pi*D*r_s*rho_g)) + 1)

Eq(Derivative(m(t), t), 4*pi*D*r_s*rho_g*log((Y_{\infty} - 1)/(Y_s - 1)))

In [191]:
display(new_ode_solve.subs(mass_derivative, answer))
display(sp.Eq(gas_velocity,gas_velocity_solve.subs(mass_derivative, answer)))

Eq(Y(r), (Y_{\infty} - 1)*exp(-r_s*log((Y_{\infty} - 1)/(Y_s - 1))/r) + 1)

Eq(u_r(r), D*r_s*log((Y_{\infty} - 1)/(Y_s - 1))/r**2)

# Temperature
There's more! Evaporation takes in heat, this cools the droplet relative to the environment. Lower temperature, slower evaporation. This needs to be solved for given out mass transport up above. Note that the thermophysical properties (density, diffusion) are going to be assumed constant. That is not great for large temperature differences (quick evaporation systems) so an averaging scheme is normally used:

$
T = T_S + \frac{1}{3}(T_{\infty}-T_S)
$

Heat conservation has a similar form to mass conservation but has an additional thermal mass term. Hot diffusing gas will carry heat away through motion as well as thermal diffusion!

$
\boldsymbol{j_Q} = - \lambda_g \rho_g \nabla T + \sum_{i=0}^{n=N} \rho_g \boldsymbol{u} h_i - \rho_g D h_i \nabla Y_i
$

Though, looking at the above, the summation ends up just being the mass flux multiplied by the component enthalpy term. We can therefore simplify as we already have the expression for the mass flux derived:

$
\boldsymbol{j_Q} = - \lambda_g \rho_g \nabla T + \sum_{i=0}^{n=N} h_i \boldsymbol{j_i}
$

Enthalpy $h$ is defined as $h = C_p dT$ but a useful trick is to notice that only differences in $h$ are used in calculations. That means that an effective $h$ can be used defined as $h = C_p T$.

In [192]:
import sympy as sp
from IPython.display import display

temperature = sp.Function("T")(position)
component_mass_fraction = sp.Function("Y_i")(position)
mass_fluxes = sp.Function("j_i")(position)
specific_heat, thermal_conductivity = sp.symbols("c_p lambda_g")
component_enthalpy = specific_heat * temperature

radial_thermal_flux =  - thermal_conductivity * temperature.diff(position) + component_enthalpy*mass_fluxes
display(radial_thermal_flux)

thermal_divergence = 1/position**2 * (position**2*radial_thermal_flux).diff(position)
display(thermal_divergence)

c_p*T(r)*j_i(r) - lambda_g*Derivative(T(r), r)

(r**2*(c_p*T(r)*Derivative(j_i(r), r) + c_p*j_i(r)*Derivative(T(r), r) - lambda_g*Derivative(T(r), (r, 2))) + 2*r*(c_p*T(r)*j_i(r) - lambda_g*Derivative(T(r), r)))/r**2

As before, the flux times the area is related to the derivative, in this case heat. We can sub in the mass derivative from before...

In [193]:
total_heat = sp.symbols("Q")
#Divergence shows that the heat flux is constant in w.r.t radial position. At the surface, heat flux times area is equal to the total rate of change of heat so this is the case at all radii.
heat = sp.Eq(4*sp.pi*position**2*radial_thermal_flux,total_heat).expand()
display(heat)

#sub in mass change instead of mass fluxes as there is a r^2 term available
m_dot = sp.symbols("m_dot")
heat_eq = heat.subs(position**2*component_enthalpy*mass_fluxes,component_enthalpy*m_dot/(4*sp.pi))
display(heat_eq)

Eq(4*pi*c_p*r**2*T(r)*j_i(r) - 4*pi*lambda_g*r**2*Derivative(T(r), r), Q)

Eq(c_p*m_dot*T(r) - 4*pi*lambda_g*r**2*Derivative(T(r), r), Q)

We can then solve for the temperature profile:

In [194]:
#Solve for temperature profile in terms of radial positon
temperature_solve = sp.dsolve(heat_eq,temperature).subs(m_dot,mass_derivative)
display(temperature_solve)



Eq(T(r), (Q + exp(c_p*(C1 - 1/(4*pi*lambda_g*r))*Derivative(m(t), t)))/(c_p*Derivative(m(t), t)))

Apply the boundary conditions as before, solving for $Q$!

In [195]:
surface_temperature, ambient_temperature = sp.symbols(r"T_s T_{\infty}")

#apply boundary conditions
thermal_constant_solve = temperature_solve.subs(position,droplet_radius).subs(temperature.subs(position,droplet_radius), surface_temperature)
thermal_constant = sp.solve(thermal_constant_solve,C1)[0]
temperature_solve = temperature_solve.subs(C1, sp.solve(thermal_constant_solve,C1)[0])
display(temperature_solve)

#far field limit
thermal_far_field = sp.limit(temperature_solve.rhs, position, sp.oo)
display(sp.Eq(ambient_temperature,thermal_far_field).simplify())

Q_solve = sp.solve(sp.Eq(ambient_temperature,thermal_far_field), total_heat)[0]
display(sp.Eq(total_heat,Q_solve))

Eq(T(r), (Q + exp(c_p*(-1/(4*pi*lambda_g*r) + log((-Q + T_s*c_p*Derivative(m(t), t))*exp(c_p*Derivative(m(t), t)/(4*pi*lambda_g*r_s)))/(c_p*Derivative(m(t), t)))*Derivative(m(t), t)))/(c_p*Derivative(m(t), t)))

Eq(T_{\infty}, (-Q*exp(c_p*Derivative(m(t), t)/(4*pi*lambda_g*r_s)) + Q + T_s*c_p*exp(c_p*Derivative(m(t), t)/(4*pi*lambda_g*r_s))*Derivative(m(t), t))/(c_p*Derivative(m(t), t)))

Eq(Q, c_p*(T_s*exp(c_p*Derivative(m(t), t)/(4*pi*lambda_g*r_s)) - T_{\infty})*Derivative(m(t), t)/(exp(c_p*Derivative(m(t), t)/(4*pi*lambda_g*r_s)) - 1))